In [ ]:
import os
from copy import deepcopy
import numpy as np
import random
import warnings

import torch
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from utils import calc_logit_norm, validate, extract_features, \
    visualize_features, get_reduced_features, add_gaussian_noise, \
    entropy_loss, collect_params, configure_model, adapt_model, \
    get_animated_features, get_partial_dataloader, animate_3d_features, logit_adjusted_adaptation,\
    class_counts

from models import CNN, CNN3
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # Replace "0" with the desired GPU device index


In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)

random_seed = 2025

torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

In [ ]:
# Parameters
LOG_FREQUENCY =  1000
DO_ADJUST_LOGITS = False
TAU = 1.0
PARTIAL_CLASSES = [9]
NOISE_SEVERITY = 5
MODEL_PATH = '/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/mnist_model.pth'

In [ ]:
def capture_params(model, param_names, captured={}):
    """
    Captures parameter values for the specified parameters in the model.

    Parameters:
        model (nn.Module): The PyTorch model.
        param_names (list of str): List of parameter names to capture. For example, ['fc1.weight', 'fc1.bias'].

    Returns:
        dict: A dictionary where keys are parameter names and values are detached copies of the parameter tensors.
    """
    for name, param in model.named_parameters():
        if name in param_names:
            # Detach and clone the parameter to avoid future modifications
            captured[name] = param.detach().clone()
    return captured

def logit_adjusted_adaptation(model, adaptation_loader, test_loader, log_frequency=100, 
                              epochs=10, class_distribution=[0.1 for i in range(10)], 
                              tau=1.0, do_adjust_logits=True):


    class_distribution = torch.Tensor(class_distribution).to('cuda')

    test_acc_list = []
    reduced_feature_list = []
    predicted_labels_list = []
    param_list = []

    deepcopy_model = deepcopy(model)
    # test accuracy
    test_accuracy,_ = validate(deepcopy_model, test_loader)
    test_acc_list.append((test_accuracy, 0))

    steps = 0
    total_steps = len(adaptation_loader) * epochs


    #### Model Configuration ####
    configure_model(model)
    bn_params, bn_names = collect_params(model, freeze_layers=["bn100"])
    print(bn_names)
    tent_optimizer = optim.Adam(bn_params, lr=1e-3)


    for epoch in range(epochs):

        for x_data, y_data in adaptation_loader:

            captured = {}

            tent_optimizer.zero_grad()
            x_data, y_data = x_data.to('cuda'), y_data.to('cuda')
            logits = model(x_data)


            # adjust logits
            if do_adjust_logits:
                adjusted_logits = logits + tau * torch.log(class_distribution + 1e-12)
            else:
                adjusted_logits = logits


            loss = entropy_loss(adjusted_logits).mean(0)
            loss.backward()

            # calculate the cosine similarity between gradients of batchnorm bias and fully connected layer weights
            # bn1_grad = model.bn1.weight.grad.detach().clone()
            # bn2_grad = model.bn2.weight.grad.detach().clone()
            bn2_beta_grad = model.bn2.bias.grad.detach().clone()
            bn2_gamma_grad = model.bn2.weight.grad.detach().clone()

            captured['bn2.bias.grad'] = bn2_beta_grad
            captured['bn2.weight.grad'] = bn2_gamma_grad

            tent_optimizer.step()

            # collect batch norm params
            param_names = ["bn1.weight", "bn1.bias", "bn2.weight", "bn2.bias"]
            param_list.append(capture_params(model, param_names, captured))

            if (steps+1) % log_frequency == 0:
                deepcopy_model = deepcopy(model)
                # test accuracy
                test_accuracy,_ = validate(deepcopy_model, test_loader)
                test_acc_list.append((test_accuracy, steps))
                print(f"Adaptation step {steps}/{total_steps}, Entropy Loss: {loss.item()}")
                print(f"Test Accuracy: {test_accuracy}")

            steps += 1

    return test_acc_list, reduced_feature_list, predicted_labels_list, param_list

In [ ]:
test_class_distribution = []
for i in range(10):
    if i in PARTIAL_CLASSES:
        test_class_distribution.append(1.0/len(PARTIAL_CLASSES))
    else:
        test_class_distribution.append(0.0)


transform = transforms.Compose([
    transforms.ToTensor(),  # converts to tensor and scales image pixel values to [0, 1]
    transforms.Normalize((0.1307,), (0.3081,))  # normalize using MNIST's mean and std
])

train_dataset = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=True, download=False, transform=transform)
test_dataset  = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=False, download=False, transform=transform)

model = torch.load(MODEL_PATH)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Noisy data
noisy_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: add_gaussian_noise(x, severity=NOISE_SEVERITY)),
    transforms.Normalize((0.1307,), (0.3081,))
])

noisy_test_dataset = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=False, download=False, transform=noisy_transform)

# when doing TTA, dataloader should be shuffled
noisy_train_loader = DataLoader(noisy_test_dataset, batch_size=64, shuffle=True)

noisy_test_loader = DataLoader(noisy_test_dataset, batch_size=64, shuffle=False)


########################################################################
# Adapt the model to balanced dataset
# test_acc_list, reduced_feature_list, _ = adapt_model(model, noisy_train_loader, noisy_test_loader, reducer, log_frequency=500, feature_layer='fc1')
# print(f"Class Counts: {class_counts(noisy_train_loader)}")
balanced_test_acc_list, _, _, balanced_param_list = logit_adjusted_adaptation(model, noisy_train_loader, noisy_test_loader, do_adjust_logits=False,
                                                                                      log_frequency=LOG_FREQUENCY, epochs=40)

########################################################################


In [ ]:
balanced_gamma_1 = [torch.norm(x['bn1.weight']).item() for x in balanced_param_list]
balanced_gamma_2 = [torch.norm(x['bn2.weight']).item() for x in balanced_param_list]
balanced_beta_1 = [torch.norm(x['bn1.bias']).item() for x in balanced_param_list]
balanced_beta_2 = [torch.norm(x['bn2.bias']).item() for x in balanced_param_list]
steps =len(balanced_param_list)


In [ ]:
# Adapt the model to Imbalanced dataset
imbalanced_test_loader = get_partial_dataloader(noisy_test_dataset, PARTIAL_CLASSES, final_samples=10000)
print(f"Class Counts: {class_counts(imbalanced_test_loader)}")

model = torch.load(MODEL_PATH)

test_acc_list, _, _, partial_param_list = logit_adjusted_adaptation(model, imbalanced_test_loader, noisy_test_loader, class_distribution=test_class_distribution,
                                                                    do_adjust_logits=False, log_frequency=LOG_FREQUENCY, tau=TAU, epochs=40)


In [ ]:
imbalanced_gamma_1 = [torch.norm(x['bn1.weight']).item() for x in partial_param_list]
imbalanced_gamma_2 = [torch.norm(x['bn2.weight']).item() for x in partial_param_list]
imbalanced_beta_1 = [torch.norm(x['bn1.bias']).item() for x in partial_param_list]
imbalanced_beta_2 = [torch.norm(x['bn2.bias']).item() for x in partial_param_list]
steps =len(partial_param_list)

In [ ]:
# plot gamma_1, gamma_2, beta_1, beta_2 in 4 subplots
from matplotlib.lines import Line2D

fig, ax = plt.subplots(2, 2)

ax[0, 0].plot(range(steps), balanced_gamma_1, label='Balanced')
ax[0, 0].plot(range(steps), imbalanced_gamma_1, label='Imbalanced')
ax[0, 0].set_title('Gamma 1')

ax[0, 1].plot(range(steps), balanced_gamma_2, label='Balanced')
ax[0, 1].plot(range(steps), imbalanced_gamma_2, label='Imbalanced')
ax[0, 1].set_title('Gamma 2')

ax[1, 0].plot(range(steps), balanced_beta_1, label='Balanced')
ax[1, 0].plot(range(steps), imbalanced_beta_1, label='Imbalanced')
ax[1, 0].set_title('Beta 1')

ax[1, 1].plot(range(steps), balanced_beta_2, label='Balanced')
ax[1, 1].plot(range(steps), imbalanced_beta_2, label='Imbalanced')
ax[1, 1].set_title('Beta 2')


fig.set_size_inches(9, 7)
legend_elements = [Line2D([0], [0], color='tab:blue',label='Balanced'),
                   Line2D([0], [0], color='tab:orange', label='Imbalanced')]
fig.legend(handles=legend_elements, loc='upper right')
plt.show()

In [ ]:
from scipy.signal import savgol_filter
import torch.nn.functional as F
# plot cosine similarity between bn gradients and fc weights for balanced and imbalanced tta

columns =5
rows = 2

fig, ax = plt.subplots(rows,columns)
fig.set_size_inches(20, 8)
fig.suptitle('Cosine Sim between beta grads and fc weights', fontsize=16)

digit = 0

balanced_beta_grad = [x[f"bn2.bias.grad"] for x in balanced_param_list]
w = torch.matmul(model.fc1.weight.T, model.fc2.weight.T)
padding_length = len(w[:,0]) - len(balanced_beta_grad[0])
padded_balanced_beta_grad = [F.pad(x, (0, padding_length), mode='constant', value=0) for x in balanced_beta_grad]

partial_beta_grad = [x[f"bn2.bias.grad"] for x in partial_param_list]
padded_partial_beta_grad = [F.pad(x, (0, padding_length), mode='constant', value=0) for x in partial_beta_grad]


for ii in range(rows):
    for jj in range(columns):
        digit = ii * columns + jj
        print(digit)
        w_c = w[:, digit]
        data = [torch.cosine_similarity(padded_balanced_beta_grad[i], w_c, dim=0).item() for i in range(len(padded_balanced_beta_grad))]   
        data = savgol_filter(data, 200, 2)
        ax[ii][jj].plot(range(steps), data, label='Balanced', color='tab:blue')


        data = [torch.cosine_similarity(padded_partial_beta_grad[i], w_c, dim=0).item() for i in range(len(padded_partial_beta_grad))]   
        data = savgol_filter(data, 200, 2)
        ax[ii][jj].plot(range(steps), data, label='Imbalanced', color='tab:orange')

        ax[ii][jj].set_title(f"sim_{digit}")
        ax[ii][jj].set_ylim(-0.1, 0.1)

fig.legend(handles=legend_elements, loc='upper right')
    

In [ ]:
columns =5
rows = 2

fig, ax = plt.subplots(rows,columns)
fig.set_size_inches(20, 8)
fig.suptitle('Cosine Sim between beta grads and fc weights', fontsize=16)

digit = 0

balanced_beta_grad = [x[f"bn2.bias.grad"] for x in balanced_param_list]
w = model.fc2.weight

partial_beta_grad = [x[f"bn2.bias.grad"] for x in partial_param_list]

for ii in range(rows):
    for jj in range(columns):
        digit = ii * columns + jj
        print(digit)
        w_c = w[digit]
        data = [torch.cosine_similarity(balanced_beta_grad[i], w_c, dim=0).item() for i in range(len(balanced_beta_grad))]   
        data = savgol_filter(data, 200, 2)
        ax[ii][jj].plot(range(steps), data, label='Balanced', color='tab:blue')


        data = [torch.cosine_similarity(partial_beta_grad[i], w_c, dim=0).item() for i in range(len(partial_beta_grad))]   
        data = savgol_filter(data, 200, 2)
        ax[ii][jj].plot(range(steps), data, label='Imbalanced', color='tab:orange')

        ax[ii][jj].set_title(f"sim_{digit}")
        ax[ii][jj].set_ylim(-0.2, 0.3)

fig.legend(handles=legend_elements, loc='upper right')

In [ ]:
# W = fc1*fc2
columns =5
rows = 2

fig, ax = plt.subplots(rows,columns)
fig.set_size_inches(20, 8)
fig.suptitle('Cosine Sim between beta and fc weights', fontsize=16)

digit = 0

balanced_beta_grad = [x[f"bn2.bias"] for x in balanced_param_list]
w = torch.matmul(model.fc1.weight.T, model.fc2.weight.T)
padding_length = len(w[:,0]) - len(balanced_beta_grad[0])
padded_balanced_beta_grad = [F.pad(x, (0, padding_length), mode='constant', value=0) for x in balanced_beta_grad]

partial_beta_grad = [x[f"bn2.bias"] for x in partial_param_list]
padded_partial_beta_grad = [F.pad(x, (0, padding_length), mode='constant', value=0) for x in partial_beta_grad]


for ii in range(rows):
    for jj in range(columns):
        digit = ii * columns + jj
        print(digit)
        w_c = w[:, digit]
        data = [torch.cosine_similarity(padded_balanced_beta_grad[i], w_c, dim=0).item() for i in range(len(padded_balanced_beta_grad))]   
        # data = savgol_filter(data, 200, 2)
        ax[ii][jj].plot(range(steps), data, label='Balanced', color='tab:blue')


        data = [torch.cosine_similarity(padded_partial_beta_grad[i], w_c, dim=0).item() for i in range(len(padded_partial_beta_grad))]   
        # data = savgol_filter(data, 200, 2)
        ax[ii][jj].plot(range(steps), data, label='Imbalanced', color='tab:orange')

        ax[ii][jj].set_title(f"sim_{digit}")
        ax[ii][jj].set_ylim(-0.1, 0.15)

fig.legend(handles=legend_elements, loc='upper right')

In [ ]:
# W = fc2
columns =5
rows = 2

fig, ax = plt.subplots(rows,columns)
fig.set_size_inches(20, 8)
fig.suptitle('Cosine Sim between beta  and fc weights', fontsize=16)

digit = 0

balanced_beta_grad = [x[f"bn2.bias"] for x in balanced_param_list]
w = model.fc2.weight

partial_beta_grad = [x[f"bn2.bias"] for x in partial_param_list]

for ii in range(rows):
    for jj in range(columns):
        digit = ii * columns + jj
        print(digit)
        w_c = w[digit]
        data = [torch.cosine_similarity(balanced_beta_grad[i], w_c, dim=0).item() for i in range(len(balanced_beta_grad))]   
        ax[ii][jj].plot(range(steps), data, label='Balanced', color='tab:blue')


        data = [torch.cosine_similarity(partial_beta_grad[i], w_c, dim=0).item() for i in range(len(partial_beta_grad))]   
        ax[ii][jj].plot(range(steps), data, label='Imbalanced', color='tab:orange')

        ax[ii][jj].set_title(f"sim_{digit}")
        ax[ii][jj].set_ylim(-0.2, 0.4)

fig.legend(handles=legend_elements, loc='upper right')

In [ ]:
# cosine similarity with beta grads and weighted sum of fc weights
all_logits = [i for i in range(10)]

visible_wc = torch.zeros(model.fc2.weight[0].shape).to('cuda')
for ii in PARTIAL_CLASSES:
    visible_wc += model.fc2.weight[ii]

visible_wc /= len(PARTIAL_CLASSES)


non_visible_wc = torch.zeros(model.fc2.weight[0].shape).to('cuda')
for ii in range(10):
    if ii not in PARTIAL_CLASSES:
        non_visible_wc += model.fc2.weight[ii]

non_visible_wc /= (10 - len(PARTIAL_CLASSES))

balanced_beta = [x[f"bn2.bias.grad"] for x in balanced_param_list]
partial_beta = [x[f"bn2.bias.grad"] for x in partial_param_list]

fig, ax = plt.subplots(1,2)
fig.set_size_inches(8, 4)

data = [torch.cosine_similarity(balanced_beta[i], visible_wc, dim=0).item() for i in range(len(balanced_beta))]
data = savgol_filter(data, 200, 2)
ax[0].plot(range(steps), data, label='Balanced', color='tab:blue')

data = [torch.cosine_similarity(partial_beta[i], visible_wc, dim=0).item() for i in range(len(partial_beta))]
data = savgol_filter(data, 200, 2)
ax[0].plot(range(steps), data, label='Imbalanced', color='tab:orange')
ax[0].set_title('Visible')
ax[0].set_ylim(-0.5,0.5)


data = [torch.cosine_similarity(balanced_beta[i], non_visible_wc, dim=0).item() for i in range(len(balanced_beta))]
data = savgol_filter(data, 200, 2)
ax[1].plot(range(steps), data, label='Balanced', color='tab:blue')
data = [torch.cosine_similarity(partial_beta[i], non_visible_wc, dim=0).item() for i in range(len(partial_beta))]
data = savgol_filter(data, 200, 2)
ax[1].plot(range(steps), data, label='Imbalanced', color='tab:orange')
ax[1].set_title('Non Visible')
ax[1].set_ylim(-0.5,0.5)

fig.legend(handles=legend_elements, loc='upper center')


In [ ]:
# cosine similarity with beta and weighted sum of fc weights
all_logits = [i for i in range(10)]

visible_wc = torch.zeros(model.fc2.weight[0].shape).to('cuda')
for ii in PARTIAL_CLASSES:
    visible_wc += model.fc2.weight[ii]

visible_wc /= len(PARTIAL_CLASSES)


non_visible_wc = torch.zeros(model.fc2.weight[0].shape).to('cuda')
for ii in range(10):
    if ii not in PARTIAL_CLASSES:
        non_visible_wc += model.fc2.weight[ii]

non_visible_wc /= (10 - len(PARTIAL_CLASSES))

balanced_beta = [x[f"bn2.bias"] for x in balanced_param_list]
partial_beta = [x[f"bn2.bias"] for x in partial_param_list]

fig, ax = plt.subplots(1,2)
fig.set_size_inches(8, 4)

data = [torch.cosine_similarity(balanced_beta[i], visible_wc, dim=0).item() for i in range(len(balanced_beta))]   
ax[0].plot(range(steps), data, label='Balanced', color='tab:blue')
data = [torch.cosine_similarity(partial_beta[i], visible_wc, dim=0).item() for i in range(len(partial_beta))]
ax[0].plot(range(steps), data, label='Imbalanced', color='tab:orange')
ax[0].set_title('Visible')
ax[0].set_ylim(-0.5,0.5)


data = [torch.cosine_similarity(balanced_beta[i], non_visible_wc, dim=0).item() for i in range(len(balanced_beta))]
ax[1].plot(range(steps), data, label='Balanced', color='tab:blue')
data = [torch.cosine_similarity(partial_beta[i], non_visible_wc, dim=0).item() for i in range(len(partial_beta))]
ax[1].plot(range(steps), data, label='Imbalanced', color='tab:orange')
ax[1].set_title('Non Visible')
ax[1].set_ylim(-0.5,0.5)

fig.legend(handles=legend_elements, loc='upper center')


In [ ]:
torch.cosine_similarity(model.fc2.weight[7],model.bn2.bias, dim=0).item()

In [ ]:
def validate_temp(model, dataloader, criterion=nn.CrossEntropyLoss()):
    predicted_labels = []
    model.eval()
    correct = 0
    total = 0
    total_loss = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to('cuda'), labels.to('cuda')
            outputs = model(images)
            # Get predictions from the maximum value
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            predicted_labels.extend(predicted.cpu().numpy())
            loss = criterion(outputs, labels)
            total_loss += loss.item()

    accuracy = correct / total
    validation_loss = total_loss / len(dataloader)
    return accuracy, validation_loss, predicted_labels

In [ ]:
_, _, predicted_labels = validate_temp(model, noisy_test_loader)

In [ ]:
from collections import Counter

Counter(predicted_labels)